<a href="https://colab.research.google.com/github/romero-sebastian/econ3916-statsml/blob/main/Lab19/Lab_19_Tree_Based_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[link text](https://)# Lab 19: Tree-Based Models — Random Forests
## ECON 3916: Data Science for Economists
### Guided Construction Lab | 30 min Core + 15 min Extension

---

**Learning Objectives:**
- Compare a single decision tree, Ridge regression, and Random Forest on the same dataset
- Tune Random Forest hyperparameters via cross-validation
- Extract and interpret feature importance (MDI and permutation)
- Apply the Ch 18 evaluation toolkit (confusion matrix, AUC) to a RF classifier

**Dataset:** California Housing (sklearn) — recurring thread from Ch 4, 12-16

---

## Setup

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load California Housing data
# -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from sklearn.inspection import permutation_importance

# Create figures directory if it doesn't exist
import os
os.makedirs('figures', exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load California Housing (you've seen this dataset since Ch 4)
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set: {X_train.shape[0]} observations, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} observations')
print(f'Features: {list(X.columns)}')

## Part 1: Decision Tree vs. Random Forest vs. Ridge (GUIDED — 10 min)

We compare three models on the same data. The question: does the Random Forest's
added complexity produce meaningfully better predictions than simpler alternatives?

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 2: Train and compare Decision Tree, Ridge, and RF
# -----------------------------------------------------------

# 1. Unrestricted Decision Tree (high variance, low bias)
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

# 2. Ridge Regression (Ch 16 callback — stable but linear)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# 3. Random Forest (100 trees, default settings)
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

# Compare all three
results = []
for name, model in [('Single Tree', tree), ('Ridge (Ch 16)', ridge), ('Random Forest', rf)]:
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Train RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Train R²': r2_score(y_train, train_pred),
        'Test R²': r2_score(y_test, test_pred),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(4))

### Interpretation Questions

1. Which model has the largest gap between Train R² and Test R²? What does this tell you about its variance?
2. The Random Forest's Test R² is substantially higher than Ridge. What does this suggest about the relationship between features and housing prices — is it linear or non-linear?
3. **Callback to Ch 15:** Where does each model sit on the bias-variance U-curve?

1. The Single Decision Tree has the largest gap between Train R squared and Test r squared. This means severe overfitting and extremely high variance. Because the tree is grown without any depth restriction, it memorizes the training data — fitting every quirk and noise point — but fails to generalize to held-out observations. This is the textbook definition of a high-variance model.

2. The Random Forest achieving a substantially higher Test R² than Ridge ( 0.81 to 0.60) shows  the relationship between the features and California housing prices is non-linear and involves interaction effects. Ridge regression, being a linear model, can only capture additive linear relationships between each feature and the target. The fact that the RF outperforms Ridge implies the data-generating process contains nonlinearities

3. On the bias-variance U-curve, the 3 models occupy different positions. The SDT sits at the far right of the curve — extremely low bias (it fits training data perfectly) but extremely high variance since it generalizes pretty bad. Ridge regression sits toward the left — higher bias because it imposes a linear functional form that may not match the true relationship, but low variance due to regularization. The random forest occupies pretty much the sweet spot near the minimum of the u curve. By averaging over many of the decorrelated trees, variance is substantially reduced for any tree while also reducing bias compared to the linear Ridge model, giving us the best test performance of the three.

## Part 2: Feature Importance — MDI vs. Permutation (GUIDED — 10 min)

Random Forests tell us which features matter most for prediction.
But remember: **predictive importance ≠ causal importance.**

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 3: Compare MDI and permutation feature importance
# -----------------------------------------------------------
mdi_importance = pd.Series(
    rf.feature_importances_, index=X.columns
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MDI importance
mdi_importance.plot.barh(ax=axes[0], color='#6B8BA4')
axes[0].set_title('MDI Feature Importance (from training)', fontsize=12)
axes[0].set_xlabel('Importance')

# Permutation importance (more reliable — uses test set)
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE
)
perm_importance = pd.Series(
    perm_result.importances_mean, index=X.columns
).sort_values(ascending=True)

perm_importance.plot.barh(ax=axes[1], color='#C47B5A')
axes[1].set_title('Permutation Importance (from test set)', fontsize=12)
axes[1].set_xlabel('Importance (decrease in R²)')

plt.tight_layout()
plt.savefig('figures/ch19_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 by MDI:', mdi_importance.tail(3).index.tolist())
print('Top 3 by Permutation:', perm_importance.tail(3).index.tolist())

### Interpretation Questions

1. Do MDI and permutation importance agree on the top features? Where do they diverge?
2. Why might MDI overstate the importance of `Latitude` and `Longitude` (hint: how many unique values do they have)?
3. **Critical question:** If `MedInc` is the top feature, does that mean raising median income in a neighborhood will raise housing prices? Why or why not? (Hint: Chapter 10 DAGs)

1. Both MDI and permutation importance broadly agree that medInc (median income) is the single most important predictor of housing prices, and both methods also agree that aveOccup (average occupancy) carries meaningful weight. The main point of divergence is in the geographic features: MDI tends to rank `Latitude` and `Longitude` more highly than permutation importance does. This divergence is methodologically meaningful — MDI is computed on training data where the model has already memorized spatial patterns, while permutation importance is computed on the test set and measures how much model performance actually degrades when a feature is scrambled, giving a more honest picture of out-of-sample relevance.

2. MDI (Mean Decrease in Impurity) favors high-cardinality features, or features with a large number of distinct values. The dataset contains thousands of different values for the continuous variables "Latitude" and "Longitude." Even if the actual predictive improvement is small, the tree algorithm can nearly always discover a split on these features that minimizes impurity because there are many potential split locations. As a result, their MDI ratings are higher than those of features with fewer unique values. Because permutation significance solely assesses whether shuffling a feature affects out-of-sample predictions, it avoids this bias because it is independent of the splits' construction.

3. No, increasing neighborhood income does not always result in higher home prices in a way that is meaningful to policy, even though medInc is the most significant predictive factor. Feature importance quantifies associations in the data rather than causal effects, as we saw with directed acyclic graphs (DAGs) in Chapter 10. Wealthier neighborhoods may concurrently have better schools, lower crime rates, better infrastructure, and proximity to economic centers, all of which jointly determine both income levels and housing demand. These factors are likely to confuse the relationship between income and housing prices. The same price effect might not always result from a legislative move that mechanically increased reported median income without altering these underlying amenities.

Predictive importance tells us what variables help the model forecast prices accurately, but causal claims require identification strategies (instrumental variables, natural experiments, etc.) that the Random Forest framework cannot provide.

## Part 3: Hyperparameter Tuning & Classification (YOUR TASK — 10 min)

Now YOU tune a Random Forest and apply it to a classification problem.

In [ ]:
# -----------------------------------------------------------
# ✎ YOUR TASK — Fill in the blanks
# Tune the Random Forest with GridSearchCV
# -----------------------------------------------------------

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 0.5, None],  # sqrt = sqrt(p), 0.5 = half, None = all
}

# GridSearchCV: 5-fold CV, optimizing negative MSE (standard sklearn convention)
# n_jobs=-1 uses all available CPU cores to parallelize the grid search
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}')

# Evaluate best model on test set
best_rf = grid_search.best_estimator_
test_rmse = np.sqrt(mean_squared_error(y_test, best_rf.predict(X_test)))
test_r2 = r2_score(y_test, best_rf.predict(X_test))
print(f'Tuned RF — Test RMSE: {test_rmse:.4f}, Test R²: {test_r2:.4f}')

In [ ]:
# -----------------------------------------------------------
# ✎ YOUR TASK — Fill in the blanks
# Classification: Predict expensive homes (> median)
# -----------------------------------------------------------

# Create binary target: 1 if price > median, 0 otherwise
median_price = np.median(y)
y_binary = (y > median_price).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_binary, test_size=0.2, random_state=RANDOM_STATE
)

# Fit RF classifier and Logistic Regression (Ch 17 callback)
rf_clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_clf.fit(X_train_c, y_train_c)

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_c, y_train_c)

# Compare AUC (Ch 18 evaluation toolkit)
for name, model in [('Logistic Regression (Ch 17)', log_reg), ('Random Forest', rf_clf)]:
    y_proba = model.predict_proba(X_test_c)[:, 1]
    auc = roc_auc_score(y_test_c, y_proba)
    print(f'{name:35s} — AUC: {auc:.4f}')

### Final Interpretation

1. How much did hyperparameter tuning improve the RF over default settings?
2. Does the RF classifier beat logistic regression on AUC? Is the difference practically significant?
3. If you were deploying this for a real estate company, which model would you choose and why? Consider: accuracy, interpretability, maintenance cost.

*Answers:*

1. Hyperparameter tuning via GridSearchCV typically yields a modest but meaningful improvement over the default Random Forest settings. The default RF uses `n_estimators=100`, `max_depth=None`, and `max_features='sqrt'`. GridSearchCV searches over combinations of deeper/shallower trees and different feature subset sizes, selecting the configuration that minimizes cross-validated MSE. In practice on this dataset, the tuned model tends to improve Test R² by roughly 1–3 percentage points over the default (e.g., from ~0.81 to ~0.82–0.84), representing a small but real gain. The more important benefit of tuning is reduced overfitting risk: limiting `max_depth` constrains tree complexity and can improve generalization even if the raw R² improvement appears small.

2. The Random Forest classifier typically achieves a meaningfully higher AUC than logistic regression on this dataset — commonly in the range of 0.93–0.95 for RF vs. 0.87–0.89 for logistic regression. This ~5–7 percentage point gap is practically significant for a binary classification task. An AUC of 0.93+ means that if you randomly sample one above-median and one below-median home, the RF correctly ranks the expensive home higher over 93% of the time. In a business context such as mortgage underwriting or property valuation, this level of discrimination could translate to meaningfully fewer misclassifications, with real financial consequences.

3. For a real estate company, the choice depends on the use case. If the primary goal is predictive accuracy — for instance, an automated valuation model used to flag pricing anomalies or support offer decisions — the Random Forest is the better choice, given its substantially higher AUC and R². However, if the model needs to be audited, explained to clients, or pass regulatory scrutiny (e.g., fair housing compliance), logistic regression has a decisive advantage: its coefficients are directly interpretable as log-odds ratios, easy to audit, and straightforward to explain to non-technical stakeholders. A pragmatic deployment strategy might use RF for internal automated scoring and logistic regression as a parallel explainability layer. On maintenance cost, RF requires more compute for retraining and hyperparameter re-tuning as data drifts, while logistic regression is lightweight and fast to update. On balance, for a sophisticated real estate firm with data science infrastructure, the RF's accuracy gains are likely worth the added complexity.

---

## Extension (Optional — 15 min): Partial Dependence Plots

Partial dependence plots show the marginal effect of a feature on the prediction,
averaging over all other features. This reveals non-linearities the RF captures.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 4: Partial dependence plots for top features
# -----------------------------------------------------------
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(12, 4))
PartialDependenceDisplay.from_estimator(
    best_rf, X_test, features=['MedInc', 'AveOccup'],
    kind='average', ax=ax
)
plt.suptitle('Partial Dependence — RF captures non-linear effects OLS misses', fontsize=13)
plt.tight_layout()
plt.savefig('figures/ch19_partial_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

### Extension Interpretation

Important non-linearities in the data-generating process that a linear model like Ridge or OLS could never capture are revealed by the partial dependence charts. The PDP for `MedInc` exhibits a traditional diminishing-returns pattern, flattening and possibly plateauing at extremely high income values after sharply expanding at low-to-moderate income levels. One linear coefficient could not adequately capture this. The PDP for `AveOccup` usually has a dramatic negative effect at low occupancy levels (highly occupied units are associated with lower prices, potentially reflecting crowding), followed by a leveling off. This is another example of a non-linear threshold effect. These results confirm that the RF performs far better than Ridge on this dataset: the genuine conditional expectation function E[Price | X] is not linear in the features, and the RF's adaptable, non-parametric tree-based structure is far better suited to approximate it.

---
## AI-Assisted Expansion: Interactive Random Forest Dashboard

Built using the P.R.I.M.E. prompt provided in the lab. The dashboard lets users adjust
`n_estimators` and `max_features` interactively and observe how model performance and
feature importance respond in real time.

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Interactive Plotly + ipywidgets dashboard
# Generated via P.R.I.M.E. prompt, reviewed and verified.
# -----------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# Cache baseline Ridge and single-tree scores for comparison panel
ridge_test_r2 = r2_score(y_test, ridge.predict(X_test))
tree_test_r2  = r2_score(y_test, tree.predict(X_test))

# --- Widgets ---
n_est_slider = widgets.IntSlider(
    value=100, min=1, max=500, step=10,
    description='n_estimators:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)

max_feat_slider = widgets.IntSlider(
    value=3, min=1, max=8, step=1,
    description='max_features:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()

def update_dashboard(change):
    """
    ipywidgets observer: fires whenever either slider changes.
    We re-fit a fresh RF inside the callback because sklearn estimators
    are stateful — changing hyperparameters requires a full refit.
    This is intentionally kept fast by using small n_estimators at low slider values.
    """
    n_est = n_est_slider.value
    max_feat = max_feat_slider.value

    # Re-fit RF with current slider values
    rf_live = RandomForestRegressor(
        n_estimators=n_est,
        max_features=max_feat,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf_live.fit(X_train, y_train)

    train_r2 = r2_score(y_train, rf_live.predict(X_train))
    test_r2  = r2_score(y_test,  rf_live.predict(X_test))

    feat_imp = pd.Series(
        rf_live.feature_importances_, index=X.columns
    ).sort_values(ascending=True)

    # Build 3-panel Plotly figure
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            'Model comparison (Test R²)',
            f'Feature importance (max_features={max_feat})',
            'Train vs Test R² (current RF)'
        ]
    )

    # Panel 1: Bar chart comparing models on test R²
    fig.add_trace(
        go.Bar(
            x=['Single Tree', 'Ridge', f'RF (n={n_est}, mf={max_feat})'],
            y=[tree_test_r2, ridge_test_r2, test_r2],
            marker_color=['#C47B5A', '#6B8BA4', '#4C956C'],
            name='Test R²'
        ),
        row=1, col=1
    )

    # Panel 2: Feature importance bar chart (updates with max_features)
    fig.add_trace(
        go.Bar(
            x=feat_imp.values,
            y=feat_imp.index,
            orientation='h',
            marker_color='#6B8BA4',
            name='MDI Importance'
        ),
        row=1, col=2
    )

    # Panel 3: Train vs Test R² comparison for current RF
    fig.add_trace(
        go.Bar(
            x=['Train R²', 'Test R²'],
            y=[train_r2, test_r2],
            marker_color=['#4C956C', '#2D6A4F'],
            name='R² split'
        ),
        row=1, col=3
    )

    fig.update_layout(
        height=420,
        title_text=f'Random Forest Dashboard — n_estimators={n_est}, max_features={max_feat}',
        showlegend=False
    )
    fig.update_yaxes(range=[0, 1], row=1, col=1)
    fig.update_yaxes(range=[0, 1], row=1, col=3)

    with output:
        output.clear_output(wait=True)
        fig.show()

# Attach observer to both sliders — fires update_dashboard on any value change
n_est_slider.observe(update_dashboard, names='value')
max_feat_slider.observe(update_dashboard, names='value')

# Render initial state
display(widgets.VBox([n_est_slider, max_feat_slider, output]))
update_dashboard(None)

### Dashboard Findings

The interactive dashboard reveals several important relationships between Random Forest hyperparameters and model performance:

- **n_estimators and variance reduction:** At very low `n_estimators` (1–10), the RF performance is volatile and often closer to a single tree. As n_estimators increases, Test R² stabilizes and improves, with diminishing returns typically setting in around 100–200 trees. Beyond ~200 estimators, additional trees add computational cost but negligible predictive gain — illustrating the law of diminishing returns in ensemble size.

- **max_features and the bias-variance tradeoff:** Smaller `max_features` values (e.g., 1–2) force each tree to use very few features per split, creating highly decorrelated trees but also increasing each individual tree's bias. Larger values (e.g., 7–8, near all features) reduce tree diversity and risk returning toward single-tree variance. The default `sqrt(p) ≈ 3` for this 8-feature dataset typically sits near the optimum.

- **Feature importance sensitivity:** As `max_features` decreases, individual features are selected less often, and importance scores become more uniform. At higher values, `MedInc` and `Latitude`/`Longitude` tend to dominate because they are almost always available as split candidates.

---
### Push to GitHub

```bash
cd econ-lab-19-random-forests
git add notebooks/ figures/ README.md verification-log.md
git commit -m "Lab 19: Random Forest vs OLS — California Housing"
git push origin main
```

Submit your GitHub repo link on Canvas.